### Score function for Gaussian prabability path

In [11]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

# 1. 定义前向/反向路径的参数演化（这里使用最经典的线性流/扩散模型参数）
# 设定 t=0 是纯噪声(原点)，t=1 是生成目标 z
def get_params(t):
    alpha_t = t          # 均值系数从 0 变到 1
    # 为了防止 t=1 时方差为 0 导致分母除以 0 报错，我们加一个极小的微量 1e-4
    beta_t_sq = (1 - t) + 1e-4  
    return alpha_t, beta_t_sq

# 2. 设定我们要生成的目标点 z (二维空间中的某个确定点)
z = np.array([3.0, 3.0])

# 3. 创建二维网格空间
x1 = np.linspace(-4, 4, 20)
x2 = np.linspace(-4, 4, 20)
X1, X2 = np.meshgrid(x1, x2)

# 为了方便矩阵运算，将网格展平
grid_points = np.stack([X1, X2], axis=-1) # 形状: (20, 20, 2)

# 4. 定义核心的绘图更新函数
def update_plot(t):
    alpha_t, beta_t_sq = get_params(t)
    
    # 计算当前时刻的均值中心
    mean_t = alpha_t * z
    
    # --- 计算概率密度分布 p_t(x|z) ---
    # 高斯分布公式中的指数部分: -||x - mu||^2 / (2 * sigma^2)
    dist_to_mean_sq = (X1 - mean_t[0])**2 + (X2 - mean_t[1])**2
    P = np.exp(-dist_to_mean_sq / (2 * beta_t_sq)) / (2 * np.pi * beta_t_sq)
    
    # --- 计算条件得分向量场 (Score Field) ---
    # 公式: ∇_x log p = (α_t * z - x) / β_t^2
    # 这里的 X1, X2 就是当前位置 x
    U = (alpha_t * z[0] - X1) / beta_t_sq
    V = (alpha_t * z[1] - X2) / beta_t_sq
    
    # --- 开始绘图 ---
    fig, ax = plt.subplots(figsize=(8, 7))
    
    # 画背景的概率密度等高线 (热力图效果)
    contour = ax.contourf(X1, X2, P, levels=20, cmap='YlOrRd', alpha=0.6)
    
    # 画向量场 (Quiver)
    # scale 参数用来控制箭头的视觉长度，不至于太密挡住视线
    # color 设定为蓝色，方便与背景热力图区分
    quiver = ax.quiver(X1, X2, U, V, color='blue', scale=30, pivot='mid', width=0.0015)
    
    # 标记目标点 z 和 当前引力中心
    ax.plot(z[0], z[1], 'g*', markersize=15, label='Target $z$')
    ax.plot(mean_t[0], mean_t[1], 'ko', markersize=8, label='Center $\\alpha_t z$')
    
    # 图表装饰
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_xlabel('$x_1$ (Dimension 1)')
    ax.set_ylabel('$x_2$ (Dimension 2)')
    ax.set_title(f'Time $t = {t:.2f}$ | $\\alpha_t={alpha_t:.2f}$ | $\\beta_t^2={beta_t_sq:.4f}$')
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.legend(loc='upper left')
    
    plt.show()

# 5. 使用 ipywidgets 创建交互滑块
# FloatSlider 允许你精细调整 t，从 0.00 到 1.00，步长 0.02
interact(
    update_plot, 
    t=FloatSlider(value=0.0, min=0.0, max=1.0, step=0.02, description='Time (t):')
);

interactive(children=(FloatSlider(value=0.0, description='Time (t):', max=1.0, step=0.02), Output()), _dom_cla…